In [1]:
"""
Audio Processing & Temporal Alignment Pipeline (bản rút gọn - giai đoạn ASR + Chunking)
------------------------------------------------------------------------------------
Tham chiếu: ndkhoa_proposal.pdf (mục 3, 4.1-4.4, 5) + demo_data.ipynb

Chức năng:
  1. Trích xuất audio từ .mp4 -> mono PCM 16kHz (ffmpeg), đúng chuẩn nêu ở mục 4.1.
  2. Chạy Faster-Whisper với VAD filter tích hợp sẵn (Silero VAD) - mục 4.2 + 4.3.
  3. Sinh transcript có mốc thời gian ở mức câu (segment) và mức từ (word) nếu bật.
  4. Làm sạch nhẹ (clean_text) mà KHÔNG dùng LLM - chỉ chuẩn hoá khoảng trắng/viết hoa/
     dấu câu cuối câu, giữ nguyên raw_text để đối chiếu (mục 4.5).
  5. Ghi ra 1 file JSON duy nhất, cấu trúc "audio-first" (chưa có scene từ mô-đun thị
     giác). Kèm sẵn hàm align_with_scenes() cài đặt công thức overlap ở mục 4.4, để khi
     có scene_boundaries.json từ nhóm thị giác thì chỉ cần gọi hàm là ra JSON theo đúng
     schema ở mục 5 của đề xuất.

Cách dùng: sửa các biến trong khối CONFIG bên dưới rồi chạy:
    python3 audio_to_json.py
Không dùng argparse theo yêu cầu — mọi đường dẫn/tham số đều khai báo trực tiếp trong file.
"""

import json
import math
import re
import subprocess
import sys
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Optional

from faster_whisper import WhisperModel

# ============================== CONFIG ======================================
# Sửa các giá trị dưới đây cho phù hợp với môi trường chạy thực tế.

# --- Đường dẫn ---
INPUT_VIDEO_PATH = "uploads/L21_V001_cut.mp4"
OUTPUT_JSON_PATH = "outputs/L21_V001_cut.json"
# File audio trung gian (wav), sẽ bị ghi đè mỗi lần chạy, có thể xoá sau khi xong.
TEMP_AUDIO_PATH = "audio_pipeline/_tmp_audio.wav"

# Đường dẫn tới ffmpeg.exe (chỉ cần thiết nếu KHÔNG cài "imageio-ffmpeg" và ffmpeg cũng
# chưa có trong PATH của hệ thống). Cách khuyến nghị: chạy
#     pip install imageio-ffmpeg
# rồi để nguyên FFMPEG_BINARY = None -> script sẽ tự tìm binary ffmpeg mà thư viện này
# tải sẵn, không cần cài ffmpeg thủ công hay đụng vào biến môi trường PATH của Windows.
# Nếu muốn chỉ định thủ công, đổi thành ví dụ: FFMPEG_BINARY = r"C:\ffmpeg\bin\ffmpeg.exe"
FFMPEG_BINARY: Optional[str] = None

# Nếu đã có scene boundary từ mô-đun thị giác (mục 4.4), trỏ tới file JSON dạng:
#   [{"scene_id": 1, "start_time": 0.0, "end_time": 12.4}, ...]
# Để None nếu chưa có -> script chỉ xuất transcript theo segment ASR gốc.
SCENES_JSON_PATH: Optional[str] = None

# --- Cấu hình model ASR (mục 3) ---
# "large-v3" cho chất lượng tốt nhất nhưng cần GPU để chạy đủ nhanh.
# Khi test trên CPU nên dùng "small"/"base"/"medium" trước, đổi lại "large-v3" khi có GPU.
MODEL_SIZE = "large-v3"
DEVICE = "cpu"              # "cuda" nếu có GPU
COMPUTE_TYPE = "int8"       # int8 tối ưu cho CPU; dùng "float16" nếu chạy GPU
LANGUAGE = "vi"             # None để auto-detect

# --- Tham số VAD (mục 4.2), dùng Silero VAD tích hợp sẵn trong faster-whisper ---
VAD_FILTER = True
VAD_MIN_SILENCE_MS = 500

# --- Tham số ASR ---
BEAM_SIZE = 5
WORD_TIMESTAMPS = True      # bật để phục vụ tách đoạn tại scene boundary (mục 4.4)

# --- Chuẩn hoá audio trước khi đưa vào model (mục 4.1) ---
TARGET_SAMPLE_RATE = 16000
TARGET_CHANNELS = 1
TARGET_LUFS = -16.0          # chuẩn hoá loudness theo EBU R128 bằng ffmpeg loudnorm

# ============================================================================


@dataclass
class WordTiming:
    word: str
    start: float
    end: float
    probability: float


@dataclass
class AudioSegment:
    segment_id: int
    start: float
    end: float
    raw_text: str
    clean_text: str
    confidence: float
    no_speech_prob: float
    words: list = field(default_factory=list)


def resolve_ffmpeg_binary() -> str:
    """Xác định đường dẫn ffmpeg thực thi được:
    1. Nếu FFMPEG_BINARY được set thủ công trong CONFIG -> dùng luôn.
    2. Ngược lại, thử lấy binary mà gói "imageio-ffmpeg" đã tải sẵn (khuyến nghị,
       không cần cài ffmpeg hệ thống, chạy tốt trên Windows/macOS/Linux).
    3. Nếu không có imageio-ffmpeg, fallback về lệnh "ffmpeg" (yêu cầu đã có trong PATH).
    """
    if FFMPEG_BINARY:
        return FFMPEG_BINARY
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except ImportError:
        print(
            "[Gợi ý] Chưa cài 'imageio-ffmpeg' và FFMPEG_BINARY đang để None -> sẽ thử "
            "gọi lệnh 'ffmpeg' từ PATH hệ thống. Nếu bị lỗi, chạy: pip install imageio-ffmpeg",
            file=sys.stderr,
        )
        return "ffmpeg"


def extract_audio(video_path: str, audio_path: str) -> None:
    """Trích xuất luồng audio từ video, chuẩn hoá về mono/16kHz/-16 LUFS (mục 4.1).

    Dùng ffmpeg trực tiếp qua subprocess thay vì load cả video vào RAM -> tiết kiệm
    bộ nhớ đáng kể với video dài, đúng tinh thần xử lý streaming/line-by-line.
    """
    Path(audio_path).parent.mkdir(parents=True, exist_ok=True)
    ffmpeg_bin = resolve_ffmpeg_binary()
    cmd = [
        ffmpeg_bin, "-y",
        "-i", video_path,
        "-vn",  # bỏ luồng video
        "-ac", str(TARGET_CHANNELS),
        "-ar", str(TARGET_SAMPLE_RATE),
        "-af", f"loudnorm=I={TARGET_LUFS}:LRA=11:TP=-1.5",
        "-acodec", "pcm_s16le",
        audio_path,
    ]
    try:
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    except FileNotFoundError as e:
        raise RuntimeError(
            f"Không tìm thấy chương trình '{ffmpeg_bin}'. Cách khắc phục nhanh nhất: chạy "
            f"'pip install imageio-ffmpeg' rồi thử lại (không cần cài ffmpeg thủ công). "
            f"Hoặc tự cài ffmpeg và sửa biến FFMPEG_BINARY ở đầu file trỏ thẳng tới ffmpeg.exe."
        ) from e
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg thất bại khi trích xuất audio:\n{result.stderr}")


def clean_text(text: str) -> str:
    """Làm sạch tối thiểu, KHÔNG dùng LLM (mục 4.5): chuẩn hoá khoảng trắng, viết hoa
    đầu câu, đảm bảo dấu câu kết thúc. Không được thêm thông tin mới vào nội dung gốc.
    """
    t = re.sub(r"\s+", " ", text).strip()
    if not t:
        return t
    t = t[0].upper() + t[1:]
    if t[-1] not in ".!?…":
        t += "."
    return t


def pseudo_confidence(avg_logprob: float) -> float:
    """faster-whisper không trả về confidence trực tiếp, chỉ có avg_logprob (log-domain).
    Quy đổi sang khoảng [0, 1] bằng exp() để dễ đọc/lọc theo threshold. Đây là ước lượng
    xấp xỉ, không phải xác suất chuẩn -> chỉ dùng để so sánh tương đối giữa các segment.
    """
    return round(min(1.0, max(0.0, math.exp(avg_logprob))), 4)


def transcribe(audio_path: str, model: WhisperModel):
    """Chạy ASR + VAD, trả về generator segment để xử lý streaming (không giữ toàn bộ
    audio trong RAM). info chứa ngôn ngữ phát hiện được và xác suất tương ứng.
    """
    segments, info = model.transcribe(
        audio_path,
        language=LANGUAGE,
        beam_size=BEAM_SIZE,
        vad_filter=VAD_FILTER,
        vad_parameters=dict(min_silence_duration_ms=VAD_MIN_SILENCE_MS),
        word_timestamps=WORD_TIMESTAMPS,
    )
    return segments, info


def build_audio_segments(raw_segments) -> list:
    """Duyệt qua generator segment một lần (streaming) và dựng cấu trúc dữ liệu chuẩn hoá.
    Không load trước toàn bộ segment vào list rồi mới xử lý -> giảm áp lực bộ nhớ với
    video dài, chỉ giữ đúng dữ liệu văn bản cần thiết.
    """
    result = []
    for idx, seg in enumerate(raw_segments, start=1):
        words = []
        if WORD_TIMESTAMPS and seg.words:
            for w in seg.words:
                words.append(WordTiming(
                    word=w.word.strip(),
                    start=round(w.start, 3),
                    end=round(w.end, 3),
                    probability=round(w.probability, 4),
                ))
        raw = seg.text.strip()
        item = AudioSegment(
            segment_id=idx,
            start=round(seg.start, 3),
            end=round(seg.end, 3),
            raw_text=raw,
            clean_text=clean_text(raw),
            confidence=pseudo_confidence(seg.avg_logprob),
            no_speech_prob=round(seg.no_speech_prob, 4),
            words=words,
        )
        result.append(item)
        # In tiến độ ngay khi xử lý xong từng đoạn, hữu ích khi theo dõi video dài.
        print(f"[{item.start:>7.2f}s -> {item.end:>7.2f}s] (conf={item.confidence}) {raw}")
    return result


def segment_to_dict(seg: AudioSegment) -> dict:
    d = {
        "segment_id": seg.segment_id,
        "time_range": {"start": seg.start, "end": seg.end},
        "audio_metadata": {
            "raw_text": seg.raw_text,
            "clean_text": seg.clean_text,
            "confidence": seg.confidence,
            "no_speech_prob": seg.no_speech_prob,
        },
    }
    if seg.words:
        d["audio_metadata"]["words"] = [
            {"word": w.word, "start": w.start, "end": w.end, "probability": w.probability}
            for w in seg.words
        ]
    return d


def compute_overlap_ratio(scene: dict, seg: AudioSegment, eps: float = 1e-6) -> float:
    """Cài đặt công thức (1)-(2) ở mục 4.4 của đề xuất:
        I_ij = max(0, min(e_i, b_j) - max(s_i, a_j))
        R_ij = I_ij / (b_j - a_j + eps)
    """
    si, ei = scene["start_time"], scene["end_time"]
    aj, bj = seg.start, seg.end
    intersection = max(0.0, min(ei, bj) - max(si, aj))
    duration = (bj - aj) + eps
    return intersection / duration


def align_with_scenes(audio_segments: list, scenes: list, overlap_threshold: float = 0.5) -> list:
    """Gán mỗi audio segment vào scene có tỷ lệ chồng lấn (R_ij) lớn nhất (mục 4.4).
    Trả về danh sách scene theo đúng schema JSON ở mục 5 của đề xuất, có
    alignment_metadata.overlap_ratio và needs_review.
    """
    scenes_out = {
        s["scene_id"]: {
            "scene_id": s["scene_id"],
            "time_range": {"start": s["start_time"], "end": s["end_time"]},
            "audio_metadata": {"raw_text": [], "clean_text": [], "confidence": []},
            "alignment_metadata": {"overlap_ratio": 0.0, "needs_review": True},
        }
        for s in scenes
    }

    for seg in audio_segments:
        best_scene, best_ratio = None, 0.0
        for scene in scenes:
            ratio = compute_overlap_ratio(scene, seg)
            if ratio > best_ratio:
                best_scene, best_ratio = scene, ratio
        if best_scene is None:
            continue
        target = scenes_out[best_scene["scene_id"]]
        target["audio_metadata"]["raw_text"].append(seg.raw_text)
        target["audio_metadata"]["clean_text"].append(seg.clean_text)
        target["audio_metadata"]["confidence"].append(seg.confidence)
        target["alignment_metadata"]["overlap_ratio"] = round(
            max(target["alignment_metadata"]["overlap_ratio"], best_ratio), 4
        )
        target["alignment_metadata"]["needs_review"] = best_ratio < overlap_threshold

    # Gộp list câu trong từng scene thành 1 chuỗi văn bản liền mạch.
    for scene in scenes_out.values():
        am = scene["audio_metadata"]
        am["raw_text"] = " ".join(am["raw_text"])
        am["clean_text"] = " ".join(am["clean_text"])
        am["confidence"] = (
            round(sum(am["confidence"]) / len(am["confidence"]), 4) if am["confidence"] else 0.0
        )

    return list(scenes_out.values())


def main():
    video_path = Path(INPUT_VIDEO_PATH)
    if not video_path.exists():
        print(f"Không tìm thấy file video: {video_path}", file=sys.stderr)
        sys.exit(1)

    print(f"--- [1/4] Trích xuất & chuẩn hoá audio từ {video_path.name} ---")
    extract_audio(str(video_path), TEMP_AUDIO_PATH)

    print(f"--- [2/4] Nạp model Faster-Whisper ({MODEL_SIZE}, {DEVICE}/{COMPUTE_TYPE}) ---")
    model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

    print("--- [3/4] Chạy ASR + VAD, sinh transcript có mốc thời gian ---")
    raw_segments, info = transcribe(TEMP_AUDIO_PATH, model)
    audio_segments = build_audio_segments(raw_segments)

    print("--- [4/4] Đóng gói kết quả ra JSON ---")
    output = {
        "source_video": str(video_path.name),
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "pipeline_config": {
            "asr_model": f"faster-whisper-{MODEL_SIZE}",
            "device": DEVICE,
            "compute_type": COMPUTE_TYPE,
            "requested_language": LANGUAGE,
            "vad_filter": VAD_FILTER,
            "word_timestamps": WORD_TIMESTAMPS,
        },
        "audio_info": {
            "duration_sec": round(info.duration, 3),
            "detected_language": info.language,
            "language_probability": round(info.language_probability, 4),
        },
        "segments": [segment_to_dict(s) for s in audio_segments],
    }

    # Nếu đã có scene boundary từ mô-đun thị giác thì gán thêm bản đã align theo scene.
    if SCENES_JSON_PATH and Path(SCENES_JSON_PATH).exists():
        with open(SCENES_JSON_PATH, "r", encoding="utf-8") as f:
            scenes = json.load(f)
        output["scenes"] = align_with_scenes(audio_segments, scenes)

    out_path = Path(OUTPUT_JSON_PATH)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    print(f"\nHoàn tất. Đã ghi {len(audio_segments)} segment vào: {out_path}")


if __name__ == "__main__":
    main()


c:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- [1/4] Trích xuất & chuẩn hoá audio từ L21_V001_cut.mp4 ---
--- [2/4] Nạp model Faster-Whisper (large-v3, cpu/int8) ---
--- [3/4] Chạy ASR + VAD, sinh transcript có mốc thời gian ---
[   4.43s ->    7.89s] (conf=0.9139) Chào mừng quý vị đến với chương trình 60 giây của đài truyền hình thành phố Hồ Chí Minh
[   7.89s ->   10.15s] (conf=0.9139) Chương trình sáng nay có những thông tin nổi bật sau đây
[  14.52s ->   18.86s] (conf=0.9139) Đồng bằng sông Cổ Long với tình trạng sụt lúng gấp gần 20 lần so với nước biển dân
[  19.62s ->   22.68s] (conf=0.9139) Vận chuyển các tóc trái tim từ Hà Nội về Huế ghép cho bệnh nhân
[  23.56s ->   27.94s] (conf=0.9139) Châu Âu trúng dọi với nhiệt độ nóng như thiêu đốt cùng những đám cháy rừng
[  31.55s ->   34.05s] (conf=0.9139) Sụt lúng đang là vấn đề cấp bách với đồng bằng sông Cổ Long
[  34.05s ->   37.39s] (conf=0.9139) khi có nơi sụt lúng trung bình lên tới 5,7 cm hột năm
[  37.39s ->   40.07s] (conf=0.9139) tức là gấp gần 20 lần so với nước biể

In [3]:
"""
Audio Processing & Temporal Alignment Pipeline (bản rút gọn - giai đoạn ASR + Chunking)
------------------------------------------------------------------------------------
Tham chiếu: ndkhoa_proposal.pdf (mục 3, 4.1-4.4, 5) + demo_data.ipynb

Chức năng:
  1. Trích xuất audio từ .mp4 -> mono PCM 16kHz (ffmpeg), đúng chuẩn nêu ở mục 4.1.
  2. Chạy Faster-Whisper với VAD filter tích hợp sẵn (Silero VAD) - mục 4.2 + 4.3.
  3. Sinh transcript có mốc thời gian ở mức câu (segment) và mức từ (word) nếu bật.
  4. Làm sạch nhẹ (clean_text) mà KHÔNG dùng LLM - chỉ chuẩn hoá khoảng trắng/viết hoa/
     dấu câu cuối câu, giữ nguyên raw_text để đối chiếu (mục 4.5).
  5. Ghi ra 1 file JSON duy nhất, cấu trúc "audio-first" (chưa có scene từ mô-đun thị
     giác). Kèm sẵn hàm align_with_scenes() cài đặt công thức overlap ở mục 4.4, để khi
     có scene_boundaries.json từ nhóm thị giác thì chỉ cần gọi hàm là ra JSON theo đúng
     schema ở mục 5 của đề xuất.

Cách dùng: sửa các biến trong khối CONFIG bên dưới rồi chạy:
    python3 audio_to_json.py
Không dùng argparse theo yêu cầu — mọi đường dẫn/tham số đều khai báo trực tiếp trong file.
"""

import json
import math
import re
import subprocess
import sys
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Optional

from faster_whisper import WhisperModel

# ============================== CONFIG ======================================
# Sửa các giá trị dưới đây cho phù hợp với môi trường chạy thực tế.

# --- Đường dẫn ---
INPUT_VIDEO_PATH = "uploads/L21_V001_cut.mp4"
OUTPUT_JSON_PATH = "outputs/L21_V001_cut_v3.json"
# File audio trung gian (wav), sẽ bị ghi đè mỗi lần chạy, có thể xoá sau khi xong.
TEMP_AUDIO_PATH = "audio_pipeline/_tmp_audio.wav"

# Đường dẫn tới ffmpeg.exe (chỉ cần thiết nếu KHÔNG cài "imageio-ffmpeg" và ffmpeg cũng
# chưa có trong PATH của hệ thống). Cách khuyến nghị: chạy
#     pip install imageio-ffmpeg
# rồi để nguyên FFMPEG_BINARY = None -> script sẽ tự tìm binary ffmpeg mà thư viện này
# tải sẵn, không cần cài ffmpeg thủ công hay đụng vào biến môi trường PATH của Windows.
# Nếu muốn chỉ định thủ công, đổi thành ví dụ: FFMPEG_BINARY = r"C:\ffmpeg\bin\ffmpeg.exe"
FFMPEG_BINARY: Optional[str] = None

# Nếu đã có scene boundary từ mô-đun thị giác (mục 4.4), trỏ tới file JSON dạng:
#   [{"scene_id": 1, "start_time": 0.0, "end_time": 12.4}, ...]
# Để None nếu chưa có -> script chỉ xuất transcript theo segment ASR gốc.
SCENES_JSON_PATH: Optional[str] = None

# --- Cấu hình model ASR (mục 3) ---
# Có thể là:
#   (a) tên model chuẩn của faster-whisper, vd "large-v3", "medium", "small"; hoặc
#   (b) đường dẫn local tới model đã convert sang CTranslate2, vd model PhoWhisper
#       (chạy convert_phowhisper.py trước để có thư mục này) -> nhận dạng tiếng Việt,
#       đặc biệt tên riêng/địa danh, thường chính xác hơn "large-v3" gốc.
# MODEL_SIZE = "large-v3"
MODEL_SIZE = r"D:\ai-challenge\AgentForce\models\phowhisper-large-ct2"   # <- bỏ comment sau khi convert PhoWhisper
DEVICE = "cpu"              # "cuda" nếu có GPU
COMPUTE_TYPE = "int8"       # int8 tối ưu cho CPU; dùng "float16" nếu chạy GPU
LANGUAGE = "vi"             # None để auto-detect

# --- Từ khoá miền dữ liệu (domain hotwords) ---
# Danh sách tên riêng/địa danh/thuật ngữ hay xuất hiện trong video, dùng để "gợi ý" cho
# decoder ưu tiên đúng chính tả thay vì từ đồng âm phổ biến hơn (vd tránh "Cửu Long" bị
# nhận nhầm thành "Cổ Long"). Không bắt buộc phải chính xác 100% các từ xuất hiện trong
# audio - chỉ cần liệt kê các từ hay bị sai để tăng xác suất decoder chọn đúng.
# Để [] nếu không cần dùng.
DOMAIN_HOTWORDS = []

# --- Tham số VAD (mục 4.2), dùng Silero VAD tích hợp sẵn trong faster-whisper ---
VAD_FILTER = True
VAD_MIN_SILENCE_MS = 500

# --- Tham số ASR ---
BEAM_SIZE = 5
WORD_TIMESTAMPS = True      # bật để phục vụ tách đoạn tại scene boundary (mục 4.4)

# --- Chuẩn hoá audio trước khi đưa vào model (mục 4.1) ---
TARGET_SAMPLE_RATE = 16000
TARGET_CHANNELS = 1
TARGET_LUFS = -16.0          # chuẩn hoá loudness theo EBU R128 bằng ffmpeg loudnorm

# ============================================================================


@dataclass
class WordTiming:
    word: str
    start: float
    end: float
    probability: float


@dataclass
class AudioSegment:
    segment_id: int
    start: float
    end: float
    raw_text: str
    clean_text: str
    confidence: float
    no_speech_prob: float
    words: list = field(default_factory=list)


def resolve_ffmpeg_binary() -> str:
    """Xác định đường dẫn ffmpeg thực thi được:
    1. Nếu FFMPEG_BINARY được set thủ công trong CONFIG -> dùng luôn.
    2. Ngược lại, thử lấy binary mà gói "imageio-ffmpeg" đã tải sẵn (khuyến nghị,
       không cần cài ffmpeg hệ thống, chạy tốt trên Windows/macOS/Linux).
    3. Nếu không có imageio-ffmpeg, fallback về lệnh "ffmpeg" (yêu cầu đã có trong PATH).
    """
    if FFMPEG_BINARY:
        return FFMPEG_BINARY
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except ImportError:
        print(
            "[Gợi ý] Chưa cài 'imageio-ffmpeg' và FFMPEG_BINARY đang để None -> sẽ thử "
            "gọi lệnh 'ffmpeg' từ PATH hệ thống. Nếu bị lỗi, chạy: pip install imageio-ffmpeg",
            file=sys.stderr,
        )
        return "ffmpeg"


def extract_audio(video_path: str, audio_path: str) -> None:
    """Trích xuất luồng audio từ video, chuẩn hoá về mono/16kHz/-16 LUFS (mục 4.1).

    Dùng ffmpeg trực tiếp qua subprocess thay vì load cả video vào RAM -> tiết kiệm
    bộ nhớ đáng kể với video dài, đúng tinh thần xử lý streaming/line-by-line.
    """
    Path(audio_path).parent.mkdir(parents=True, exist_ok=True)
    ffmpeg_bin = resolve_ffmpeg_binary()
    cmd = [
        ffmpeg_bin, "-y",
        "-i", video_path,
        "-vn",  # bỏ luồng video
        "-ac", str(TARGET_CHANNELS),
        "-ar", str(TARGET_SAMPLE_RATE),
        "-af", f"loudnorm=I={TARGET_LUFS}:LRA=11:TP=-1.5",
        "-acodec", "pcm_s16le",
        audio_path,
    ]
    try:
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    except FileNotFoundError as e:
        raise RuntimeError(
            f"Không tìm thấy chương trình '{ffmpeg_bin}'. Cách khắc phục nhanh nhất: chạy "
            f"'pip install imageio-ffmpeg' rồi thử lại (không cần cài ffmpeg thủ công). "
            f"Hoặc tự cài ffmpeg và sửa biến FFMPEG_BINARY ở đầu file trỏ thẳng tới ffmpeg.exe."
        ) from e
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg thất bại khi trích xuất audio:\n{result.stderr}")


def clean_text(text: str) -> str:
    """Làm sạch tối thiểu, KHÔNG dùng LLM (mục 4.5): chuẩn hoá khoảng trắng, viết hoa
    đầu câu, đảm bảo dấu câu kết thúc. Không được thêm thông tin mới vào nội dung gốc.
    """
    t = re.sub(r"\s+", " ", text).strip()
    if not t:
        return t
    t = t[0].upper() + t[1:]
    if t[-1] not in ".!?…":
        t += "."
    return t


def pseudo_confidence(avg_logprob: float) -> float:
    """faster-whisper không trả về confidence trực tiếp, chỉ có avg_logprob (log-domain).
    Quy đổi sang khoảng [0, 1] bằng exp() để dễ đọc/lọc theo threshold. Đây là ước lượng
    xấp xỉ, không phải xác suất chuẩn -> chỉ dùng để so sánh tương đối giữa các segment.
    """
    return round(min(1.0, max(0.0, math.exp(avg_logprob))), 4)


def transcribe(audio_path: str, model: WhisperModel):
    """Chạy ASR + VAD, trả về generator segment để xử lý streaming (không giữ toàn bộ
    audio trong RAM). info chứa ngôn ngữ phát hiện được và xác suất tương ứng.
    """
    hotwords = ", ".join(DOMAIN_HOTWORDS) if DOMAIN_HOTWORDS else None
    segments, info = model.transcribe(
        audio_path,
        language=LANGUAGE,
        beam_size=BEAM_SIZE,
        vad_filter=VAD_FILTER,
        vad_parameters=dict(min_silence_duration_ms=VAD_MIN_SILENCE_MS),
        word_timestamps=WORD_TIMESTAMPS,
        hotwords=hotwords,  # boost các từ trong DOMAIN_HOTWORDS mà không ảnh hưởng
                             # tới toàn bộ prompt/temperature reset như initial_prompt
    )
    return segments, info


def build_audio_segments(raw_segments) -> list:
    """Duyệt qua generator segment một lần (streaming) và dựng cấu trúc dữ liệu chuẩn hoá.
    Không load trước toàn bộ segment vào list rồi mới xử lý -> giảm áp lực bộ nhớ với
    video dài, chỉ giữ đúng dữ liệu văn bản cần thiết.
    """
    result = []
    for idx, seg in enumerate(raw_segments, start=1):
        words = []
        if WORD_TIMESTAMPS and seg.words:
            for w in seg.words:
                words.append(WordTiming(
                    word=w.word.strip(),
                    start=round(w.start, 3),
                    end=round(w.end, 3),
                    probability=round(w.probability, 4),
                ))
        raw = seg.text.strip()
        item = AudioSegment(
            segment_id=idx,
            start=round(seg.start, 3),
            end=round(seg.end, 3),
            raw_text=raw,
            clean_text=clean_text(raw),
            confidence=pseudo_confidence(seg.avg_logprob),
            no_speech_prob=round(seg.no_speech_prob, 4),
            words=words,
        )
        result.append(item)
        # In tiến độ ngay khi xử lý xong từng đoạn, hữu ích khi theo dõi video dài.
        print(f"[{item.start:>7.2f}s -> {item.end:>7.2f}s] (conf={item.confidence}) {raw}")
    return result


def segment_to_dict(seg: AudioSegment) -> dict:
    d = {
        "segment_id": seg.segment_id,
        "time_range": {"start": seg.start, "end": seg.end},
        "audio_metadata": {
            "raw_text": seg.raw_text,
            "clean_text": seg.clean_text,
            "confidence": seg.confidence,
            "no_speech_prob": seg.no_speech_prob,
        },
    }
    if seg.words:
        d["audio_metadata"]["words"] = [
            {"word": w.word, "start": w.start, "end": w.end, "probability": w.probability}
            for w in seg.words
        ]
    return d


def compute_overlap_ratio(scene: dict, seg: AudioSegment, eps: float = 1e-6) -> float:
    """Cài đặt công thức (1)-(2) ở mục 4.4 của đề xuất:
        I_ij = max(0, min(e_i, b_j) - max(s_i, a_j))
        R_ij = I_ij / (b_j - a_j + eps)
    """
    si, ei = scene["start_time"], scene["end_time"]
    aj, bj = seg.start, seg.end
    intersection = max(0.0, min(ei, bj) - max(si, aj))
    duration = (bj - aj) + eps
    return intersection / duration


def align_with_scenes(audio_segments: list, scenes: list, overlap_threshold: float = 0.5) -> list:
    """Gán mỗi audio segment vào scene có tỷ lệ chồng lấn (R_ij) lớn nhất (mục 4.4).
    Trả về danh sách scene theo đúng schema JSON ở mục 5 của đề xuất, có
    alignment_metadata.overlap_ratio và needs_review.
    """
    scenes_out = {
        s["scene_id"]: {
            "scene_id": s["scene_id"],
            "time_range": {"start": s["start_time"], "end": s["end_time"]},
            "audio_metadata": {"raw_text": [], "clean_text": [], "confidence": []},
            "alignment_metadata": {"overlap_ratio": 0.0, "needs_review": True},
        }
        for s in scenes
    }

    for seg in audio_segments:
        best_scene, best_ratio = None, 0.0
        for scene in scenes:
            ratio = compute_overlap_ratio(scene, seg)
            if ratio > best_ratio:
                best_scene, best_ratio = scene, ratio
        if best_scene is None:
            continue
        target = scenes_out[best_scene["scene_id"]]
        target["audio_metadata"]["raw_text"].append(seg.raw_text)
        target["audio_metadata"]["clean_text"].append(seg.clean_text)
        target["audio_metadata"]["confidence"].append(seg.confidence)
        target["alignment_metadata"]["overlap_ratio"] = round(
            max(target["alignment_metadata"]["overlap_ratio"], best_ratio), 4
        )
        target["alignment_metadata"]["needs_review"] = best_ratio < overlap_threshold

    # Gộp list câu trong từng scene thành 1 chuỗi văn bản liền mạch.
    for scene in scenes_out.values():
        am = scene["audio_metadata"]
        am["raw_text"] = " ".join(am["raw_text"])
        am["clean_text"] = " ".join(am["clean_text"])
        am["confidence"] = (
            round(sum(am["confidence"]) / len(am["confidence"]), 4) if am["confidence"] else 0.0
        )

    return list(scenes_out.values())


def main():
    video_path = Path(INPUT_VIDEO_PATH)
    if not video_path.exists():
        print(f"Không tìm thấy file video: {video_path}", file=sys.stderr)
        sys.exit(1)

    print(f"--- [1/4] Trích xuất & chuẩn hoá audio từ {video_path.name} ---")
    extract_audio(str(video_path), TEMP_AUDIO_PATH)

    print(f"--- [2/4] Nạp model Faster-Whisper ({MODEL_SIZE}, {DEVICE}/{COMPUTE_TYPE}) ---")
    model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

    print("--- [3/4] Chạy ASR + VAD, sinh transcript có mốc thời gian ---")
    raw_segments, info = transcribe(TEMP_AUDIO_PATH, model)
    audio_segments = build_audio_segments(raw_segments)

    print("--- [4/4] Đóng gói kết quả ra JSON ---")
    output = {
        "source_video": str(video_path.name),
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "pipeline_config": {
            "asr_model": f"faster-whisper-{MODEL_SIZE}",
            "device": DEVICE,
            "compute_type": COMPUTE_TYPE,
            "requested_language": LANGUAGE,
            "vad_filter": VAD_FILTER,
            "word_timestamps": WORD_TIMESTAMPS,
            "domain_hotwords": DOMAIN_HOTWORDS,
        },
        "audio_info": {
            "duration_sec": round(info.duration, 3),
            "detected_language": info.language,
            "language_probability": round(info.language_probability, 4),
        },
        "segments": [segment_to_dict(s) for s in audio_segments],
    }

    # Nếu đã có scene boundary từ mô-đun thị giác thì gán thêm bản đã align theo scene.
    if SCENES_JSON_PATH and Path(SCENES_JSON_PATH).exists():
        with open(SCENES_JSON_PATH, "r", encoding="utf-8") as f:
            scenes = json.load(f)
        output["scenes"] = align_with_scenes(audio_segments, scenes)

    out_path = Path(OUTPUT_JSON_PATH)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    print(f"\nHoàn tất. Đã ghi {len(audio_segments)} segment vào: {out_path}")


if __name__ == "__main__":
    main()


--- [1/4] Trích xuất & chuẩn hoá audio từ L21_V001_cut.mp4 ---
--- [2/4] Nạp model Faster-Whisper (D:\ai-challenge\AgentForce\models\phowhisper-large-ct2, cpu/int8) ---
--- [3/4] Chạy ASR + VAD, sinh transcript có mốc thời gian ---
[   4.43s ->   38.07s] (conf=0.9734) chào mừng quý vị đến với chương trình sáu mươi giây của đài truyền hình thành phố hồ chí minh chương trình sáng nay có những thông tin nổi bật sau đây đồng bằng sông cửu long với tình trạng sụt lún gấp gần hai mươi lần so với nước biển dần vẫn chuyển các tốc trái tim từ hà nội về huế ghép cho bệnh nhân châu âu chúng chọi với nhiệt độ nóng như thiêu đốt cùng những đám cháy rực sụt lún đang là vấn đề cấp bách với đồng bằng sông cửu long khi có nơi sụt lún trung bình lên tới năm phẩy bảy xen ti mét một năm tức là
[  38.07s ->   69.08s] (conf=0.9812) gấp gần hai mươi lần so với nước biển dần dự báo phần lớn diện tích có thể sẽ nằm dưới mực nước biển trung bình vào cuối thế kỷ hai mươi mốt chiều ngày ba mươi mốt tháng bảy tại 

In [4]:
"""
Audio Processing & Temporal Alignment Pipeline (bản rút gọn - giai đoạn ASR + Chunking)
------------------------------------------------------------------------------------
Tham chiếu: ndkhoa_proposal.pdf (mục 3, 4.1-4.4, 5) + demo_data.ipynb

Chức năng:
  1. Trích xuất audio từ .mp4 -> mono PCM 16kHz (ffmpeg), đúng chuẩn nêu ở mục 4.1.
  2. Chạy Faster-Whisper với VAD filter tích hợp sẵn (Silero VAD) - mục 4.2 + 4.3.
  3. Sinh transcript có mốc thời gian ở mức câu (segment) và mức từ (word) nếu bật.
  4. Làm sạch nhẹ (clean_text) mà KHÔNG dùng LLM - chỉ chuẩn hoá khoảng trắng/viết hoa/
     dấu câu cuối câu, giữ nguyên raw_text để đối chiếu (mục 4.5).
  5. Ghi ra JSON theo batch: mỗi video trong INPUT_DIR -> 1 file .json cùng tên trong
     OUTPUT_DIR, cấu trúc "audio-first" (chưa có scene từ mô-đun thị giác). Kèm sẵn hàm
     align_with_scenes() cài đặt công thức overlap ở mục 4.4, để khi có
     scene_boundaries.json (đặt trong SCENES_DIR, cùng tên với video) từ nhóm thị giác
     thì script tự gộp thêm khối "scenes" theo đúng schema mục 5 của đề xuất.

Cách dùng: sửa các biến trong khối CONFIG bên dưới rồi chạy:
    python3 audio_to_json.py
Không dùng argparse theo yêu cầu — mọi đường dẫn/tham số đều khai báo trực tiếp trong file.
"""

import json
import math
import re
import subprocess
import sys
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Optional

from faster_whisper import WhisperModel

# ============================== CONFIG ======================================
# Sửa các giá trị dưới đây cho phù hợp với môi trường chạy thực tế.

# --- Đường dẫn ---
# Đọc TẤT CẢ video trong INPUT_DIR (kể cả thư mục con nếu RECURSIVE=True), mỗi video
# sinh ra 1 file .json cùng tên trong OUTPUT_DIR.
INPUT_DIR = "uploads"
OUTPUT_DIR = "outputs/transcripts"
RECURSIVE = False                    # True nếu muốn quét cả thư mục con
VIDEO_EXTENSIONS = {".mp4", ".mkv", ".mov", ".avi", ".webm"}
SKIP_EXISTING = True                 # bỏ qua video đã có sẵn file .json tương ứng trong
                                      # OUTPUT_DIR -> tiện chạy lại khi bị gián đoạn giữa
                                      # chừng mà không phải xử lý lại từ đầu.

# File audio trung gian (wav) dùng chung cho mỗi lượt xử lý, bị ghi đè liên tục -> không
# cần tạo file riêng cho từng video, tiết kiệm dung lượng đĩa.
TEMP_AUDIO_PATH = "audio_pipeline/_tmp_audio.wav"

# Đường dẫn tới ffmpeg.exe (chỉ cần thiết nếu KHÔNG cài "imageio-ffmpeg" và ffmpeg cũng
# chưa có trong PATH của hệ thống). Cách khuyến nghị: chạy
#     pip install imageio-ffmpeg
# rồi để nguyên FFMPEG_BINARY = None -> script sẽ tự tìm binary ffmpeg mà thư viện này
# tải sẵn, không cần cài ffmpeg thủ công hay đụng vào biến môi trường PATH của Windows.
# Nếu muốn chỉ định thủ công, đổi thành ví dụ: FFMPEG_BINARY = r"C:\ffmpeg\bin\ffmpeg.exe"
FFMPEG_BINARY: Optional[str] = None

# Nếu đã có scene boundary từ mô-đun thị giác (mục 4.4) cho TỪNG video, đặt các file JSON
# trong SCENES_DIR với tên trùng stem video, vd video "L21_V001.mp4" -> scene file
# "L21_V001.json" trong SCENES_DIR, định dạng:
#   [{"scene_id": 1, "start_time": 0.0, "end_time": 12.4}, ...]
# Để None nếu chưa có -> script chỉ xuất transcript theo segment ASR gốc cho mọi video.
SCENES_DIR: Optional[str] = None

# --- Cấu hình model ASR (mục 3) ---
# Có thể là:
#   (a) tên model chuẩn của faster-whisper, vd "large-v3", "medium", "small"; hoặc
#   (b) đường dẫn local tới model đã convert sang CTranslate2, vd model PhoWhisper
#       (chạy convert_phowhisper.py trước để có thư mục này) -> nhận dạng tiếng Việt,
#       đặc biệt tên riêng/địa danh, thường chính xác hơn "large-v3" gốc.
MODEL_SIZE = "models/phowhisper-large-ct2"
# MODEL_SIZE = r"./models/phowhisper-large-ct2"   # <- bỏ comment sau khi convert PhoWhisper
DEVICE = "cpu"              # "cuda" nếu có GPU
COMPUTE_TYPE = "int8"       # int8 tối ưu cho CPU; dùng "float16" nếu chạy GPU
LANGUAGE = "vi"             # None để auto-detect

# --- Từ khoá miền dữ liệu (domain hotwords) ---
# Danh sách tên riêng/địa danh/thuật ngữ hay xuất hiện trong video, dùng để "gợi ý" cho
# decoder ưu tiên đúng chính tả thay vì từ đồng âm phổ biến hơn (vd tránh "Cửu Long" bị
# nhận nhầm thành "Cổ Long"). Không bắt buộc phải chính xác 100% các từ xuất hiện trong
# audio - chỉ cần liệt kê các từ hay bị sai để tăng xác suất decoder chọn đúng.
# Để [] nếu không cần dùng.
DOMAIN_HOTWORDS = []

# --- Tham số VAD (mục 4.2), dùng Silero VAD tích hợp sẵn trong faster-whisper ---
VAD_FILTER = True
VAD_MIN_SILENCE_MS = 500

# --- Tham số ASR ---
BEAM_SIZE = 5
WORD_TIMESTAMPS = True      # bật để phục vụ tách đoạn tại scene boundary (mục 4.4)

# --- Chuẩn hoá audio trước khi đưa vào model (mục 4.1) ---
TARGET_SAMPLE_RATE = 16000
TARGET_CHANNELS = 1
TARGET_LUFS = -16.0          # chuẩn hoá loudness theo EBU R128 bằng ffmpeg loudnorm

# ============================================================================


@dataclass
class WordTiming:
    word: str
    start: float
    end: float
    probability: float


@dataclass
class AudioSegment:
    segment_id: int
    start: float
    end: float
    raw_text: str
    clean_text: str
    confidence: float
    no_speech_prob: float
    words: list = field(default_factory=list)


def resolve_ffmpeg_binary() -> str:
    """Xác định đường dẫn ffmpeg thực thi được:
    1. Nếu FFMPEG_BINARY được set thủ công trong CONFIG -> dùng luôn.
    2. Ngược lại, thử lấy binary mà gói "imageio-ffmpeg" đã tải sẵn (khuyến nghị,
       không cần cài ffmpeg hệ thống, chạy tốt trên Windows/macOS/Linux).
    3. Nếu không có imageio-ffmpeg, fallback về lệnh "ffmpeg" (yêu cầu đã có trong PATH).
    """
    if FFMPEG_BINARY:
        return FFMPEG_BINARY
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except ImportError:
        print(
            "[Gợi ý] Chưa cài 'imageio-ffmpeg' và FFMPEG_BINARY đang để None -> sẽ thử "
            "gọi lệnh 'ffmpeg' từ PATH hệ thống. Nếu bị lỗi, chạy: pip install imageio-ffmpeg",
            file=sys.stderr,
        )
        return "ffmpeg"


def extract_audio(video_path: str, audio_path: str) -> None:
    """Trích xuất luồng audio từ video, chuẩn hoá về mono/16kHz/-16 LUFS (mục 4.1).

    Dùng ffmpeg trực tiếp qua subprocess thay vì load cả video vào RAM -> tiết kiệm
    bộ nhớ đáng kể với video dài, đúng tinh thần xử lý streaming/line-by-line.
    """
    Path(audio_path).parent.mkdir(parents=True, exist_ok=True)
    ffmpeg_bin = resolve_ffmpeg_binary()
    cmd = [
        ffmpeg_bin, "-y",
        "-i", video_path,
        "-vn",  # bỏ luồng video
        "-ac", str(TARGET_CHANNELS),
        "-ar", str(TARGET_SAMPLE_RATE),
        "-af", f"loudnorm=I={TARGET_LUFS}:LRA=11:TP=-1.5",
        "-acodec", "pcm_s16le",
        audio_path,
    ]
    try:
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    except FileNotFoundError as e:
        raise RuntimeError(
            f"Không tìm thấy chương trình '{ffmpeg_bin}'. Cách khắc phục nhanh nhất: chạy "
            f"'pip install imageio-ffmpeg' rồi thử lại (không cần cài ffmpeg thủ công). "
            f"Hoặc tự cài ffmpeg và sửa biến FFMPEG_BINARY ở đầu file trỏ thẳng tới ffmpeg.exe."
        ) from e
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg thất bại khi trích xuất audio:\n{result.stderr}")


def clean_text(text: str) -> str:
    """Làm sạch tối thiểu, KHÔNG dùng LLM (mục 4.5): chuẩn hoá khoảng trắng, viết hoa
    đầu câu, đảm bảo dấu câu kết thúc. Không được thêm thông tin mới vào nội dung gốc.
    """
    t = re.sub(r"\s+", " ", text).strip()
    if not t:
        return t
    t = t[0].upper() + t[1:]
    if t[-1] not in ".!?…":
        t += "."
    return t


def pseudo_confidence(avg_logprob: float) -> float:
    """faster-whisper không trả về confidence trực tiếp, chỉ có avg_logprob (log-domain).
    Quy đổi sang khoảng [0, 1] bằng exp() để dễ đọc/lọc theo threshold. Đây là ước lượng
    xấp xỉ, không phải xác suất chuẩn -> chỉ dùng để so sánh tương đối giữa các segment.
    """
    return round(min(1.0, max(0.0, math.exp(avg_logprob))), 4)


def transcribe(audio_path: str, model: WhisperModel):
    """Chạy ASR + VAD, trả về generator segment để xử lý streaming (không giữ toàn bộ
    audio trong RAM). info chứa ngôn ngữ phát hiện được và xác suất tương ứng.
    """
    hotwords = ", ".join(DOMAIN_HOTWORDS) if DOMAIN_HOTWORDS else None
    segments, info = model.transcribe(
        audio_path,
        language=LANGUAGE,
        beam_size=BEAM_SIZE,
        vad_filter=VAD_FILTER,
        vad_parameters=dict(min_silence_duration_ms=VAD_MIN_SILENCE_MS),
        word_timestamps=WORD_TIMESTAMPS,
        hotwords=hotwords,  # boost các từ trong DOMAIN_HOTWORDS mà không ảnh hưởng
                             # tới toàn bộ prompt/temperature reset như initial_prompt
    )
    return segments, info


def build_audio_segments(raw_segments) -> list:
    """Duyệt qua generator segment một lần (streaming) và dựng cấu trúc dữ liệu chuẩn hoá.
    Không load trước toàn bộ segment vào list rồi mới xử lý -> giảm áp lực bộ nhớ với
    video dài, chỉ giữ đúng dữ liệu văn bản cần thiết.
    """
    result = []
    for idx, seg in enumerate(raw_segments, start=1):
        words = []
        if WORD_TIMESTAMPS and seg.words:
            for w in seg.words:
                words.append(WordTiming(
                    word=w.word.strip(),
                    start=round(w.start, 3),
                    end=round(w.end, 3),
                    probability=round(w.probability, 4),
                ))
        raw = seg.text.strip()
        item = AudioSegment(
            segment_id=idx,
            start=round(seg.start, 3),
            end=round(seg.end, 3),
            raw_text=raw,
            clean_text=clean_text(raw),
            confidence=pseudo_confidence(seg.avg_logprob),
            no_speech_prob=round(seg.no_speech_prob, 4),
            words=words,
        )
        result.append(item)
        # In tiến độ ngay khi xử lý xong từng đoạn, hữu ích khi theo dõi video dài.
        print(f"[{item.start:>7.2f}s -> {item.end:>7.2f}s] (conf={item.confidence}) {raw}")
    return result


def segment_to_dict(seg: AudioSegment) -> dict:
    d = {
        "segment_id": seg.segment_id,
        "time_range": {"start": seg.start, "end": seg.end},
        "audio_metadata": {
            "raw_text": seg.raw_text,
            "clean_text": seg.clean_text,
            "confidence": seg.confidence,
            "no_speech_prob": seg.no_speech_prob,
        },
    }
    if seg.words:
        d["audio_metadata"]["words"] = [
            {"word": w.word, "start": w.start, "end": w.end, "probability": w.probability}
            for w in seg.words
        ]
    return d


def compute_overlap_ratio(scene: dict, seg: AudioSegment, eps: float = 1e-6) -> float:
    """Cài đặt công thức (1)-(2) ở mục 4.4 của đề xuất:
        I_ij = max(0, min(e_i, b_j) - max(s_i, a_j))
        R_ij = I_ij / (b_j - a_j + eps)
    """
    si, ei = scene["start_time"], scene["end_time"]
    aj, bj = seg.start, seg.end
    intersection = max(0.0, min(ei, bj) - max(si, aj))
    duration = (bj - aj) + eps
    return intersection / duration


def align_with_scenes(audio_segments: list, scenes: list, overlap_threshold: float = 0.5) -> list:
    """Gán mỗi audio segment vào scene có tỷ lệ chồng lấn (R_ij) lớn nhất (mục 4.4).
    Trả về danh sách scene theo đúng schema JSON ở mục 5 của đề xuất, có
    alignment_metadata.overlap_ratio và needs_review.
    """
    scenes_out = {
        s["scene_id"]: {
            "scene_id": s["scene_id"],
            "time_range": {"start": s["start_time"], "end": s["end_time"]},
            "audio_metadata": {"raw_text": [], "clean_text": [], "confidence": []},
            "alignment_metadata": {"overlap_ratio": 0.0, "needs_review": True},
        }
        for s in scenes
    }

    for seg in audio_segments:
        best_scene, best_ratio = None, 0.0
        for scene in scenes:
            ratio = compute_overlap_ratio(scene, seg)
            if ratio > best_ratio:
                best_scene, best_ratio = scene, ratio
        if best_scene is None:
            continue
        target = scenes_out[best_scene["scene_id"]]
        target["audio_metadata"]["raw_text"].append(seg.raw_text)
        target["audio_metadata"]["clean_text"].append(seg.clean_text)
        target["audio_metadata"]["confidence"].append(seg.confidence)
        target["alignment_metadata"]["overlap_ratio"] = round(
            max(target["alignment_metadata"]["overlap_ratio"], best_ratio), 4
        )
        target["alignment_metadata"]["needs_review"] = best_ratio < overlap_threshold

    # Gộp list câu trong từng scene thành 1 chuỗi văn bản liền mạch.
    for scene in scenes_out.values():
        am = scene["audio_metadata"]
        am["raw_text"] = " ".join(am["raw_text"])
        am["clean_text"] = " ".join(am["clean_text"])
        am["confidence"] = (
            round(sum(am["confidence"]) / len(am["confidence"]), 4) if am["confidence"] else 0.0
        )

    return list(scenes_out.values())


def find_videos(input_dir: Path) -> list:
    """Liệt kê tất cả file video trong input_dir theo VIDEO_EXTENSIONS, sắp xếp theo tên
    để thứ tự xử lý ổn định, có thể dự đoán được giữa các lần chạy.
    """
    pattern_iter = input_dir.rglob("*") if RECURSIVE else input_dir.glob("*")
    videos = sorted(
        p for p in pattern_iter
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    )
    return videos


def load_scenes_for(video_path: Path) -> Optional[list]:
    """Tìm file scene boundary tương ứng (cùng stem) trong SCENES_DIR, nếu có."""
    if not SCENES_DIR:
        return None
    scene_path = Path(SCENES_DIR) / f"{video_path.stem}.json"
    if not scene_path.exists():
        return None
    with open(scene_path, "r", encoding="utf-8") as f:
        return json.load(f)


def process_video(video_path: Path, model: WhisperModel) -> dict:
    """Chạy toàn bộ pipeline (trích xuất audio -> ASR -> đóng gói) cho 1 video, trả về
    dict sẵn sàng json.dump. Tách riêng khỏi main() để dùng lại được cho cả chế độ batch
    lẫn khi cần gọi từ nơi khác (vd unit test, xử lý song song sau này).
    """
    print(f"  [1/3] Trích xuất & chuẩn hoá audio ...")
    extract_audio(str(video_path), TEMP_AUDIO_PATH)

    print(f"  [2/3] Chạy ASR + VAD ...")
    raw_segments, info = transcribe(TEMP_AUDIO_PATH, model)
    audio_segments = build_audio_segments(raw_segments)

    print(f"  [3/3] Đóng gói JSON ...")
    output = {
        "source_video": video_path.name,
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "pipeline_config": {
            "asr_model": f"faster-whisper-{MODEL_SIZE}",
            "device": DEVICE,
            "compute_type": COMPUTE_TYPE,
            "requested_language": LANGUAGE,
            "vad_filter": VAD_FILTER,
            "word_timestamps": WORD_TIMESTAMPS,
            "domain_hotwords": DOMAIN_HOTWORDS,
        },
        "audio_info": {
            "duration_sec": round(info.duration, 3),
            "detected_language": info.language,
            "language_probability": round(info.language_probability, 4),
        },
        "segments": [segment_to_dict(s) for s in audio_segments],
    }

    scenes = load_scenes_for(video_path)
    if scenes:
        output["scenes"] = align_with_scenes(audio_segments, scenes)

    return output


def main():
    input_dir = Path(INPUT_DIR)
    output_dir = Path(OUTPUT_DIR)
    if not input_dir.exists():
        print(f"Không tìm thấy thư mục input: {input_dir}", file=sys.stderr)
        sys.exit(1)
    output_dir.mkdir(parents=True, exist_ok=True)

    videos = find_videos(input_dir)
    if not videos:
        print(f"Không tìm thấy video nào ({', '.join(sorted(VIDEO_EXTENSIONS))}) trong {input_dir}")
        return

    print(f"Tìm thấy {len(videos)} video trong {input_dir}\n")
    print(f"Nạp model Faster-Whisper ({MODEL_SIZE}, {DEVICE}/{COMPUTE_TYPE}) ...")
    # Chỉ nạp model 1 lần rồi tái sử dụng cho toàn bộ batch -> tránh tốn thời gian/RAM
    # nạp lại model cho từng video, vốn là chi phí lớn hơn nhiều so với ASR từng file.
    model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

    done, skipped, failed = 0, 0, 0
    for idx, video_path in enumerate(videos, start=1):
        out_path = output_dir / f"{video_path.stem}.json"
        print(f"[{idx}/{len(videos)}] {video_path.name}")

        if SKIP_EXISTING and out_path.exists():
            print(f"  -> Bỏ qua (đã có {out_path.name})\n")
            skipped += 1
            continue

        try:
            result = process_video(video_path, model)
        except Exception as e:
            print(f"  -> LỖI: {e}\n", file=sys.stderr)
            failed += 1
            continue

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"  -> Đã ghi {len(result['segments'])} segment vào {out_path.name}\n")
        done += 1

    print(
        f"Hoàn tất batch: {done} thành công, {skipped} bỏ qua (đã tồn tại), "
        f"{failed} lỗi, trên tổng {len(videos)} video."
    )


if __name__ == "__main__":
    main()


Tìm thấy 3 video trong uploads

Nạp model Faster-Whisper (models/phowhisper-large-ct2, cpu/int8) ...
[1/3] L21_V001.mp4
  [1/3] Trích xuất & chuẩn hoá audio ...
  [2/3] Chạy ASR + VAD ...
[   4.43s ->   38.07s] (conf=0.9731) chào mừng quý vị đến với chương trình sáu mươi giây của đài truyền hình thành phố hồ chí minh chương trình sáng nay có những thông tin nổi bật sau đây đồng bằng sông cửu long với tình trạng sụt lún gấp gần hai mươi lần so với nước biển dần vẫn chuyển các tốc trái tim từ hà nội về huế ghép cho bệnh nhân châu âu chúng chọi với nhiệt độ nóng như thiêu đốt cùng những đám cháy rực sụt lún đang là vấn đề cấp bách với đồng bằng sông cửu long khi có nơi sụt lún trung bình lên tới năm phẩy bảy xen ti mét một năm tức là
[  38.07s ->   69.08s] (conf=0.9821) gấp gần hai mươi lần so với nước biển dần dự báo phần lớn diện tích có thể sẽ nằm dưới mực nước biển trung bình vào cuối thế kỷ hai mươi mốt chiều ngày ba mươi mốt tháng bảy tại hà nội cơ quan chuyên môn đã báo cáo lãnh đạ